In [1]:
import psycopg2
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

def show_table_sizes():
    try:
        # Access the environment variables
        DB_USERNAME = os.getenv("DB_USERNAME")
        DB_PASSWORD = os.getenv("DB_PASSWORD")
        DB_HOST = os.getenv("DB_HOST")
        DB_PORT = os.getenv("DB_PORT")
        DB_NAME = os.getenv("DB_NAME")

        # Establish a connection to the PostgreSQL database
        conn = psycopg2.connect(
            host=DB_HOST,
            port=DB_PORT,
            database=DB_NAME,
            user=DB_USERNAME,
            password=DB_PASSWORD
        )

        # Create a cursor object
        cursor = conn.cursor()

        # Query to retrieve table sizes
        cursor.execute("""
            SELECT table_name,
                   pg_size_pretty(total_bytes) AS total,
                   pg_size_pretty(index_bytes) AS index,
                   pg_size_pretty(toast_bytes) AS toast,
                   pg_size_pretty(table_bytes) AS table
            FROM (
                SELECT table_name,
                       pg_total_relation_size('public.' || table_name) AS total_bytes,
                       pg_indexes_size('public.' || table_name) AS index_bytes,
                       pg_total_relation_size('public.' || table_name) - pg_indexes_size('public.' || table_name) AS toast_bytes,
                       pg_table_size('public.' || table_name) AS table_bytes
                FROM information_schema.tables
                WHERE table_schema = 'public'
            ) AS table_sizes;
        """)

        # Fetch all table sizes
        table_sizes = cursor.fetchall()

        # Close the cursor and the connection
        cursor.close()
        conn.close()

        # Display table sizes
        print("Table Sizes:")
        for row in table_sizes:
            print(f"Table: {row[0]}, Total Size: {row[1]}, Index Size: {row[2]}, Toast Size: {row[3]}, Table Size: {row[4]}")

    except Exception as e:
        # Log the error and display a more informative message
        error_message = f"An error occurred: {str(e)}"
        print(error_message)

if __name__ == "__main__":
    show_table_sizes()


Table Sizes:
Table: total_precipitation_aladin, Total Size: 17 MB, Index Size: 16 kB, Toast Size: 17 MB, Table Size: 17 MB
Table: large-scale_snowfall_-_water_equivalent_(accumulation)_icond2, Total Size: 400 kB, Index Size: 16 kB, Toast Size: 384 kB, Table Size: 384 kB
Table: base_reflectivity_(cmax)_icond2, Total Size: 17 MB, Index Size: 16 kB, Toast Size: 17 MB, Table Size: 17 MB
Table: lat_lon_schema, Total Size: 48 kB, Index Size: 16 kB, Toast Size: 32 kB, Table Size: 32 kB
Table: maximum_wind_10m_icond2, Total Size: 213 MB, Index Size: 16 kB, Toast Size: 213 MB, Table Size: 213 MB
Table: total_precipitation_icond2, Total Size: 70 MB, Index Size: 16 kB, Toast Size: 70 MB, Table Size: 70 MB
Table: 10_metre_v_wind_component_icond2, Total Size: 39 MB, Index Size: 16 kB, Toast Size: 39 MB, Table Size: 39 MB
Table: 2_metre_temperature_icond2, Total Size: 243 MB, Index Size: 16 kB, Toast Size: 243 MB, Table Size: 243 MB
Table: total_precipitation, Total Size: 2040 kB, Index Size: 16 kB,